In [6]:
import lm_eval
import json
import random
import os
from dotenv import load_dotenv
from datasets import load_dataset
from lm_eval.tasks import TaskManager


In [7]:
original_dataset = load_dataset("PennyK98/protipa_exams_dataset", split="test")
print(original_dataset)

Dataset({
    features: ['unique_id', 'subject', 'school_level', 'year', 'series', 'label_id', 'question', 'input', 'choices', 'answer', 'answer_index', 'multimodality', 'image_path', 'mark', 'question_type', 'exercise_type', 'source_file'],
    num_rows: 330
})


In [17]:
def filter_dataset(original_dataset):
    filtered_dataset = []
    subjects = ['ΓΛΩΣΣΑ', 'ΜΑΘΗΜΑΤΙΚΑ']
    
    for item in original_dataset:
        # 1. Ελέγχουμε αν το μάθημα είναι μέσα στη λίστα μας
        # 2. Ελέγχουμε αν είναι Multiple Choice
        # 3. Ελέγχουμε αν υπάρχει μόνο μία απάντηση (όχι κόμμα στο answer_index)
        if (item['subject'] in subjects and 
            item['exercise_type'] == 'Multiple Choice' and 
            ',' not in str(item['answer_index'])):
            
            filtered_dataset.append(item)
            
    return filtered_dataset

In [ ]:
#%env LMEVAL_LOG_LEVEL=DEBUG
#!lm_eval \
    #--model hf \
    #--model_args "pretrained=EleutherAI/pythia-2.8b,dtype=float32,device=cpu" \
    #--include_path "C:\Users\panag\Desktop\ΙΕΛ ΕΡΓΑΣΙΑ\Εξετάσεις_γλωσσομάθειας\Πρότυπα_Πειραματικά" \
    #--tasks greek_protipa_exams \
    #--limit 10

In [18]:
# 1. Φόρτωση κλειδιών από το .env
load_dotenv()
api_key = os.getenv("LITELLM_ILSP_EVAL_API_KEY")
api_base = os.getenv("LITELLM_HOST")

# 2. Ορισμός των Environment Variables που περιμένει το LiteLLM
os.environ["OPENAI_API_KEY"] = api_key
os.environ["OPENAI_API_BASE"] = api_base

In [19]:
yaml_path = r'C:\Users\panag\Desktop\ΙΕΛ ΕΡΓΑΣΙΑ\Εξετάσεις_γλωσσομάθειας\Πρότυπα_Πειραματικά'
json_full_path = os.path.join(yaml_path, 'pilot_data.json')
task_manager = TaskManager(include_path=yaml_path)
model_to_test = "ilsp/Llama-Krikri-8B-Instruct"

In [20]:
all_filtered = filter_dataset(original_dataset)

# Διαλέγουμε 100 τυχαία δείγματα
pilot_100 = random.sample(all_filtered, min(len(all_filtered), 100))

# ΑΥΤΟΜΑΤΗ ΑΠΟΘΗΚΕΥΣΗ: Δημιουργεί το αρχείο μόνο του κάθε φορά
json_path = os.path.join(yaml_path, 'pilot_data.json')
with open('pilot_data.json', 'w', encoding='utf-8') as f:
    json.dump(pilot_100, f, ensure_ascii=False, indent=4)

In [21]:
yaml_string = {
    "task": "greek_protipa_exams",
    "dataset_path": "json",
    "dataset_kwargs": {
        "data_files": json_full_path
    },
    "test_split": "train",
    "output_type": "multiple_choice",
    "doc_to_text": "{% if input %}{{input}}\n{% endif %}Ερώτηση: {{question}}\nΑπάντηση:",
    "doc_to_choice": "{{choices}}",
    "doc_to_target": "{{ (answer_index | string).split(',')[0] | int }}",
    "metric_list": [
        {"metric": "acc", "aggregation": "mean", "higher_is_better": True},
        {"metric": "acc_norm", "aggregation": "mean", "higher_is_better": True}
    ]
}

In [22]:
results = lm_eval.simple_evaluate(
    model="local-completions", # Χρησιμοποιούμε completions για API κλήσεις
    model_args=f"model={model_to_test},base_url={api_base},num_fewshot=0",
    tasks=[yaml_string],
    task_manager=task_manager,
    limit=100,
    batch_size=1
)

print(results['results'])

pretrained=model=ilsp/Llama-Krikri-8B-Instruct,base_url=http://ec2-3-19-37-251.us-east-2.compute.amazonaws.com:4000,num_fewshot=0 appears to
        be an instruct or chat variant but chat template is not applied. Recommend setting `apply_chat_template` (optionally
        `fewshot_as_multiturn`).


Generating train split: 0 examples [00:00, ? examples/s]

100%|██████████| 100/100 [00:00<00:00, 381.97it/s]


ConnectTimeout: HTTPConnectionPool(host='ec2-3-19-37-251.us-east-2.compute.amazonaws.com', port=4000): Max retries exceeded with url: / (Caused by ConnectTimeoutError(<urllib3.connection.HTTPConnection object at 0x0000024F187C57B0>, 'Connection to ec2-3-19-37-251.us-east-2.compute.amazonaws.com timed out. (connect timeout=None)'))